# Module C, sequence 2: saving and loading (exercises)

Replace each `...` by what its comment asks, then run the cell. The steps follow the TP 2 of the handout of the module C. The corrected notebook is in the branch `correction-module-c`.

In [ ]:
import numpy as np
from pathlib import Path
from pymodaq_utils.math_utils import gauss1D
from pymodaq_data.data import Axis, DataRaw, DataToExport
from pymodaq_data.h5modules.saving import H5SaverLowLevel
from pymodaq_data.h5modules.data_saving import DataToExportSaver, DataLoader

DATA = Path('../data')

# the objects of the sequence 1
wavelength = Axis('wavelength', 'nm', data=np.linspace(500, 560, 121), index=0)
signal = 3 * gauss1D(wavelength.get_data(), 528, 4) + 0.1
line = DataRaw('line', data=[signal], axes=[wavelength], labels=['photodiode'], units='V')
line.errors = [np.full(121, 0.05)]
mesure = DataToExport('mesure', data=[line])

## 1. Write the file

Then open `line.h5` in H5Browser (`h5browser` in a terminal): plot the line and read its attributes.

In [ ]:
path = Path('line.h5')
path.unlink(missing_ok=True)  # init_file appends to an existing file

h5saver = H5SaverLowLevel()
...  # create the file at `path`
...  # add a detector group under '/RawData', titled 'Photodiode line'
...  # save `mesure` in the group '/RawData/Detector000' (hint: DataToExportSaver)
for node in ...:  # walk all the nodes of the file, from '/'
    print(node)
h5saver.close_file()

## 2. Read it back and compare

In [ ]:
loader = ...  # open the file for reading
loaded = ...  # load everything under '/RawData/Detector000' (a DataToExport)
loader.close_file()
back = ...  # the data named 'line'
print(...)  # is it equal to `line`? Print also its axes and its errors

## 3. Read the continuous recording of the DAQ_Viewer (enlargeable arrays)

Use your own recording (Do Save in the DAQ_Viewer of the spectrometer), or the file of the `data` folder.

In [ ]:
loader = DataLoader(DATA / 'continuous_spectra.h5')  # or the path of your own recording
...  # print the nodes of the file and find the enlargeable array (EnlData)
spectra = ...  # load this array with its navigation axis (hint: load_data, load_all=True)
loader.close_file()
print(spectra.shape[0], 'spectra recorded, shape', spectra.shape)
timestamps = ...  # the axis of the dimension 0 (hint: get_axis_from_index)
print(timestamps.label, timestamps.units, timestamps.get_data()[:3])

## 4. Mean and standard deviation of the series

In [ ]:
mean_spectrum = ...  # the mean over the navigation axis (the acquisitions), as a data object
std_spectrum = spectra.data[0].std(axis=0, ddof=1)
wavelength = ...  # the values of the signal axis (the wavelengths)
i_peak = int(np.argmax(mean_spectrum.data[0]))
print(wavelength[i_peak], mean_spectrum.data[0][i_peak], std_spectrum[i_peak])

## Control question 2

Why does the file of the continuous recording not have the same layout as a snapshot file, and what would we lose
by saving each spectrum in a separate file?